# Reconhecimento de Objetos via Webcam com MediaPipe e OpenCV

Este notebook utiliza o **Google MediaPipe Tasks** para realizar o reconhecimento de objetos em tempo real usando a sua webcam.

### Requisitos:
- `mediapipe`
- `opencv-python`

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import os
import requests

## 1. Configuração do Modelo

O MediaPipe requer um arquivo de modelo `.tflite`. Vamos baixar o `efficientdet_lite0` caso ele não exista na pasta.

In [2]:
model_path = 'efficientdet_lite0.tflite'
url = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float16/1/efficientdet_lite0.tflite"

if not os.path.exists(model_path):
    print("Baixando modelo...")
    r = requests.get(url)
    with open(model_path, 'wb') as f:
        f.write(r.content)
    print("Download concluído!")
else:
    print("Modelo já existe.")

Modelo já existe.


## 2. Inicialização do Detector

Configuramos o detector para rodar em modo de stream de vídeo.

In [3]:
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.ObjectDetectorOptions(base_options=base_options,
                                        score_threshold=0.5)
detector = vision.ObjectDetector.create_from_options(options)

## 3. Loop da Webcam

**Dica:** Pressione a tecla **'q'** para fechar a janela da webcam.

In [4]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Converter o frame para o formato do MediaPipe
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Realizar a detecção
    detection_result = detector.detect(mp_image)

    # Desenhar os resultados no frame
    for detection in detection_result.detections:
        # Caixa delimitadora
        bbox = detection.bounding_box
        start_point = bbox.origin_x, bbox.origin_y
        end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
        cv2.rectangle(frame, start_point, end_point, (0, 255, 0), 2)

        # Rótulo e score
        category = detection.categories[0]
        label = f"{category.category_name} ({category.score:.2f})"
        cv2.putText(frame, label, (bbox.origin_x, bbox.origin_y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Mostrar o frame
    cv2.imshow('MediaPipe Object Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()